# 03 — Dual OCR extraction (PaddleOCR + DeepSeek-OCR)

**Phase 3a/b/c** (plan §6 Stage 3). Runs after `02_preprocessing.ipynb`.

Goal: take every preprocessed body page (`(:PAGE {mode='ocr', role='body', preprocessedImageUri IS NOT NULL})`)
and run **two independent OCR engines** over it:

1. **PaddleOCR PP-OCRv5** (local, CPU) — `apps.backend.ocr.paddle.PaddleOCREngine`.
   Routed per page: `lang='ch'` (default, classical/modern Chinese) or
   `lang='japan'` (kanbun / Japanese-authored editorial works). Works
   **offline** after first run (models cached under `~/.paddleocr/`).
2. **DeepSeek-OCR via Silra** — `apps.backend.ocr.silra_deepseek.deepseek_ocr_page`.
   Requires **internet**; uses the OpenAI-compatible chat endpoint with
   a language-specific system prompt (classical-Chinese or kanbun).

Each engine writes its result back to the PAGE node under a distinct
property prefix (`paddleOcr*` vs `deepseekOcr*`); the fusion step that
produces the final `textFused` lives in `03b_fusion.ipynb`.

**Engine toggle** (`RUN_PADDLE` / `RUN_DEEPSEEK` env vars):

| Scenario | `RUN_PADDLE` | `RUN_DEEPSEEK` |
|---|---|---|
| Offline / flight / battery | `1` (default) | `0` |
| Online + quota | `1` | `1` |
| Online-only / re-run after PaddleOCR | `0` | `1` |

Both runners are **idempotent + resumable**: re-running picks up where
the last attempt stopped, by checking `(:PAGE).paddleOcrStatus` and
`(:PAGE).deepseekOcrStatus`. Pass `RECOMPUTE=1` to re-OCR pages that
already have a result.

**Smoke vs full run** (`RUN_FULL` env var):

- `RUN_FULL=0` (default): caps the page count at `MAX_PAGES` (default 5)
  so the notebook returns in <2 min. Use for development.
- `RUN_FULL=1`: process **every** eligible page. Use
  `scripts/run_paddle_ocr.py` (with `caffeinate -dimsu`) for long
  background runs — the notebook is meant for interactive exploration.

**Inputs**

- `notebooks/_artifacts/02_preprocessing/preprocessing.json`
- Live Neo4j + MinIO with `preprocessedImageUri` populated.
- (DeepSeek branch only) `.env` with `LLM_API_KEY` + `LLM_BASE_URL`.

**Outputs**

- `notebooks/_artifacts/03_dual_extraction/extraction.json` — per-engine
  coverage, per-page benchmark, sample transcriptions, corpus runtime
  projections.
- Neo4j: PAGEs carry `paddleOcr*` and/or `deepseekOcr*` properties.

**Next**: `03b_fusion.ipynb` (character-level align-and-vote + post-OCR
language detection).

In [12]:
from __future__ import annotations

import json
import logging
import os
import statistics
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / 'pyproject.toml').exists(), f'cannot locate repo root from {Path.cwd()}'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

loaded = load_dotenv(REPO_ROOT / '.env')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')
logging.getLogger('neo4j.notifications').setLevel(logging.WARNING)
logging.getLogger('apps.backend.pipeline.extract').setLevel(logging.INFO)

ARTIFACT_DIR = REPO_ROOT / 'notebooks' / '_artifacts' / '03_dual_extraction'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PRIOR = REPO_ROOT / 'notebooks' / '_artifacts' / '02_preprocessing' / 'preprocessing.json'

RUN_FULL = os.getenv('RUN_FULL', '0') == '1'
MAX_PAGES = int(os.getenv('MAX_PAGES', '0' if RUN_FULL else '5'))
MAX_PAGES = None if RUN_FULL else MAX_PAGES
RUN_PADDLE = os.getenv('RUN_PADDLE', '1') == '1'
RUN_DEEPSEEK = os.getenv('RUN_DEEPSEEK', '0') == '1'
RECOMPUTE = os.getenv('RECOMPUTE', '0') == '1'
DOCUMENT_ID = os.getenv('DOCUMENT_ID') or None
ROLES = tuple(r.strip() for r in os.getenv('ROLES', 'body').split(',') if r.strip())

print(f'repo root      : {REPO_ROOT}')
print(f'.env loaded    : {loaded}')
print(f'artifact dir   : {ARTIFACT_DIR}')
print(f'RUN_FULL       : {RUN_FULL}  (MAX_PAGES={MAX_PAGES})')
print(f'RUN_PADDLE     : {RUN_PADDLE}')
print(f'RUN_DEEPSEEK   : {RUN_DEEPSEEK}')
print(f'RECOMPUTE      : {RECOMPUTE}')
print(f'ROLES          : {ROLES}')
print(f'DOCUMENT_ID    : {DOCUMENT_ID!r}')

repo root      : /Users/mohasani/Ancient
.env loaded    : True
artifact dir   : /Users/mohasani/Ancient/notebooks/_artifacts/03_dual_extraction
RUN_FULL       : False  (MAX_PAGES=5)
RUN_PADDLE     : True
RUN_DEEPSEEK   : False
RECOMPUTE      : False
ROLES          : ('body',)
DOCUMENT_ID    : None


In [13]:
if not PRIOR.exists():
    raise RuntimeError(
        f'Missing prior artifact: {PRIOR}. Run 02_preprocessing.ipynb first.'
    )
prior_data = json.loads(PRIOR.read_text())
summary = prior_data.get('summary', {})
print('Phase 2 status (from prior artifact):')
print(json.dumps(summary, indent=2, ensure_ascii=False)[:1200])

Phase 2 status (from prior artifact):
{}


In [14]:
from apps.backend.graph.neo4j_client import get_driver
from apps.backend.graph.schema import init_schema
from apps.backend.storage.minio_client import get_minio_client

driver = get_driver()
minio_client = get_minio_client()
bucket = os.getenv('MINIO_BUCKET_PAGES', 'ancient-pages')

# Refresh schema so the new Phase-3 lookup indexes are present.
schema_report = init_schema(driver)
phase3_indexes = [
    i for i in schema_report['lookup_indexes']
    if i['name'].startswith('page_paddle_ocr') or
       i['name'].startswith('page_deepseek_ocr') or
       i['name'].startswith('page_fusion')
]
print('Phase-3 indexes:')
for idx in phase3_indexes:
    print(f'  - {idx["name"]}: {idx["label"]}.{idx["property"]} ({idx["state"]})')
if schema_report['errors']:
    print('SCHEMA ERRORS:', schema_report['errors'])

Phase-3 indexes:
  - page_deepseek_ocr_status_index: PAGE.deepseekOcrStatus (ONLINE)
  - page_fusion_status_index: PAGE.fusionStatus (ONLINE)
  - page_paddle_ocr_status_index: PAGE.paddleOcrStatus (ONLINE)


In [15]:
with driver.session() as session:
    counts = session.run('''
        MATCH (p:PAGE)
        WHERE p.mode = 'ocr' AND p.role IN $roles
        WITH p,
             CASE WHEN p.preprocessedImageUri IS NOT NULL THEN 'preprocessed'
                  ELSE 'unprocessed' END AS prep,
             coalesce(p.paddleOcrStatus, '(unset)') AS paddle_status,
             coalesce(p.deepseekOcrStatus, '(unset)') AS deepseek_status
        RETURN prep, paddle_status, deepseek_status, count(*) AS n
        ORDER BY prep, paddle_status, deepseek_status
    ''', roles=list(ROLES)).data()

for row in counts:
    print(row)

total_eligible = sum(r['n'] for r in counts if r['prep'] == 'preprocessed')
print(f'\nTotal preprocessed pages eligible for OCR: {total_eligible}')
if not RECOMPUTE:
    pending_paddle = sum(r['n'] for r in counts if r['prep'] == 'preprocessed' and r['paddle_status'] in ('(unset)', 'failed'))
    pending_deepseek = sum(r['n'] for r in counts if r['prep'] == 'preprocessed' and r['deepseek_status'] in ('(unset)', 'failed'))
    print(f'  pending PaddleOCR    : {pending_paddle}')
    print(f'  pending DeepSeek-OCR : {pending_deepseek}')

{'prep': 'preprocessed', 'paddle_status': 'empty', 'deepseek_status': '(unset)', 'n': 12}
{'prep': 'preprocessed', 'paddle_status': 'empty', 'deepseek_status': 'empty', 'n': 4}
{'prep': 'preprocessed', 'paddle_status': 'empty', 'deepseek_status': 'ok', 'n': 6}
{'prep': 'preprocessed', 'paddle_status': 'ok', 'deepseek_status': '(unset)', 'n': 1490}
{'prep': 'preprocessed', 'paddle_status': 'ok', 'deepseek_status': 'empty', 'n': 8}
{'prep': 'preprocessed', 'paddle_status': 'ok', 'deepseek_status': 'failed', 'n': 413}
{'prep': 'preprocessed', 'paddle_status': 'ok', 'deepseek_status': 'ok', 'n': 380}
{'prep': 'unprocessed', 'paddle_status': '(unset)', 'deepseek_status': '(unset)', 'n': 629}

Total preprocessed pages eligible for OCR: 2313
  pending PaddleOCR    : 0
  pending DeepSeek-OCR : 1915


## 1. PaddleOCR (offline-capable)

Loads PP-OCRv5 once (warmup ~30–80 s on first run; cached afterwards),
then OCRs each preprocessed page. Per-page wall-clock is recorded so we
can project full-corpus runtime.

In [16]:
paddle_report = None
if RUN_PADDLE:
    from apps.backend.ocr.paddle import PaddleOCREngine
    from apps.backend.pipeline.extract import run_paddle_pages

    print('Loading PaddleOCR engine (first time downloads ~30 MB of models)...')
    t0 = time.monotonic()
    paddle_engine = PaddleOCREngine(model_size='mobile', device='cpu')
    paddle_engine.warmup(langs=['ch'])
    print(f'Warmup done in {time.monotonic() - t0:.1f}s')

    print(f'\nRunning PaddleOCR over up to {MAX_PAGES or "ALL"} pages...')
    paddle_report = run_paddle_pages(
        driver=driver,
        minio_client=minio_client,
        engine=paddle_engine,
        bucket=bucket,
        document_id=DOCUMENT_ID,
        max_pages=MAX_PAGES,
        recompute_existing=RECOMPUTE,
        roles=ROLES,
        progress_every=10,
    )
    print('\nPaddleOCR run report:')
    print(json.dumps({k: v for k, v in paddle_report.to_dict().items() if k != 'sample_outcomes'}, indent=2, ensure_ascii=False))
else:
    print('RUN_PADDLE=0 — skipping PaddleOCR. (Set RUN_PADDLE=1 to enable.)')

2026-05-19 00:26:28,026 INFO apps.backend.ocr.paddle: Loading PaddleOCR engine lang=ch version=2.10.0 kwargs={'use_angle_cls': True, 'show_log': False, 'use_gpu': False}


Loading PaddleOCR engine (first time downloads ~30 MB of models)...
Warmup done in 0.4s

Running PaddleOCR over up to 5 pages...

PaddleOCR run report:
{
  "engine": "paddleocr",
  "pages_total": 0,
  "pages_processed": 0,
  "pages_failed": 0,
  "pages_skipped": 0,
  "total_chars": 0,
  "duration_seconds": 0.039,
  "avg_seconds_per_page": 0.0,
  "by_document": {},
  "errors": []
}


In [17]:
if RUN_PADDLE and paddle_report:
    print('PaddleOCR sample outcomes:')
    for o in paddle_report.sample_outcomes[:5]:
        print(f'  - {o["page_id"]}: {o["status"]}, {o["char_count"]} chars, '
              f'conf={o["confidence"]:.3f}, {o["duration_seconds"]:.2f}s')
    
    if paddle_report.sample_outcomes:
        first_id = paddle_report.sample_outcomes[0]['page_id']
        with driver.session() as session:
            row = session.run(
                'MATCH (p:PAGE {id: $id}) RETURN p.paddleOcrText AS text', id=first_id
            ).single()
        if row and row['text']:
            print(f'\n--- {first_id} (first 600 chars) ---')
            print(row['text'][:600])

PaddleOCR sample outcomes:


## 2. DeepSeek-OCR via Silra (requires internet)

Hits Silra's OpenAI-compatible chat endpoint with the page image
base64-encoded as a `image_url` content part + a classical-Chinese or
kanbun system prompt. Latency includes network round-trip; per-page
median is typically 8–15 s for 200-DPI scans.

In [18]:
deepseek_report = None
if RUN_DEEPSEEK:
    from apps.backend.llm.silra import get_silra_client
    from apps.backend.pipeline.extract import run_deepseek_pages

    silra_client = get_silra_client(timeout=180.0)
    print(f'Running DeepSeek-OCR over up to {MAX_PAGES or "ALL"} pages '
          f'(model={os.getenv("OCR_LLM_MODEL", "deepseek-ocr")})...')
    deepseek_report = run_deepseek_pages(
        driver=driver,
        minio_client=minio_client,
        silra_client=silra_client,
        bucket=bucket,
        document_id=DOCUMENT_ID,
        max_pages=MAX_PAGES,
        recompute_existing=RECOMPUTE,
        roles=ROLES,
        progress_every=10,
        max_tokens=4096,
        timeout=180.0,
    )
    print('\nDeepSeek-OCR run report:')
    print(json.dumps({k: v for k, v in deepseek_report.to_dict().items() if k != 'sample_outcomes'}, indent=2, ensure_ascii=False))
else:
    print('RUN_DEEPSEEK=0 — skipping DeepSeek-OCR. (Set RUN_DEEPSEEK=1 to enable.)')

RUN_DEEPSEEK=0 — skipping DeepSeek-OCR. (Set RUN_DEEPSEEK=1 to enable.)


In [19]:
if RUN_DEEPSEEK and deepseek_report:
    print('DeepSeek-OCR sample outcomes:')
    for o in deepseek_report.sample_outcomes[:5]:
        print(f'  - {o["page_id"]}: {o["status"]}, {o["char_count"]} chars, '
              f'{o["duration_seconds"]:.2f}s, error={o["error"]}')
    
    if deepseek_report.sample_outcomes:
        first_id = deepseek_report.sample_outcomes[0]['page_id']
        with driver.session() as session:
            row = session.run(
                'MATCH (p:PAGE {id: $id}) RETURN p.deepseekOcrText AS text', id=first_id
            ).single()
        if row and row['text']:
            print(f'\n--- {first_id} (first 600 chars) ---')
            print(row['text'][:600])

## 3. Coverage summary + runtime projections

Roll up per-engine status counts (across the whole graph, not just
this run) and project corpus-wide runtime from the per-page wall-clock
this notebook just measured. Useful for sizing the next background run.

In [20]:
from apps.backend.pipeline.extract import extraction_summary

extract_summary = extraction_summary(driver)
print('Corpus-wide extraction coverage:')
print(json.dumps(extract_summary, indent=2, ensure_ascii=False))

with driver.session() as session:
    counts = session.run('''
        MATCH (p:PAGE)
        WHERE p.mode='ocr' AND p.role='body'
        RETURN
          count(p) AS total_body,
          count(p.preprocessedImageUri) AS preprocessed,
          count(CASE WHEN p.paddleOcrStatus='ok' THEN 1 END) AS paddle_ok,
          count(CASE WHEN p.deepseekOcrStatus='ok' THEN 1 END) AS deepseek_ok
    ''').single()
totals = dict(counts) if counts else {}
print('\nCorpus totals:')
for k, v in totals.items():
    print(f'  {k:>14}: {v}')

Corpus-wide extraction coverage:
{
  "by_engine_status": {
    "paddleocr": {
      "empty": 22,
      "ok": 2291
    },
    "deepseek_ocr": {
      "(unset)": 1502,
      "ok": 386,
      "empty": 12,
      "failed": 413
    }
  },
  "by_tier": {
    "primary": {
      "paddleocr": {
        "empty": 1,
        "ok": 24
      },
      "deepseek_ocr": {
        "(unset)": 25
      }
    },
    "secondary": {
      "paddleocr": {
        "ok": 2267,
        "empty": 21
      },
      "deepseek_ocr": {
        "ok": 386,
        "(unset)": 1477,
        "empty": 12,
        "failed": 413
      }
    }
  },
  "total_ocr_pages": 2313
}

Corpus totals:
      total_body: 2942
    preprocessed: 2313
       paddle_ok: 2291
     deepseek_ok: 386


In [21]:
def _project(durations_seconds, total_pages, label):
    if not durations_seconds:
        return None
    median = statistics.median(durations_seconds)
    mean = statistics.mean(durations_seconds)
    return {
        'engine': label,
        'sample_size': len(durations_seconds),
        'per_page_seconds_median': round(median, 3),
        'per_page_seconds_mean': round(mean, 3),
        'total_pages': total_pages,
        'projected_hours_median': round(median * total_pages / 3600, 2),
        'projected_hours_mean': round(mean * total_pages / 3600, 2),
    }

projections = []
total_target = totals.get('preprocessed', 0)

if paddle_report and paddle_report.sample_outcomes:
    p_durations = [o['duration_seconds'] for o in paddle_report.sample_outcomes if o['status']=='ok']
    proj = _project(p_durations, total_target, 'paddleocr')
    if proj:
        projections.append(proj)
        print(f'PaddleOCR projection over {total_target} preprocessed pages:')
        print(f'  per-page median {proj["per_page_seconds_median"]}s  →  {proj["projected_hours_median"]}h total')
        print(f'  per-page mean   {proj["per_page_seconds_mean"]}s  →  {proj["projected_hours_mean"]}h total')

if deepseek_report and deepseek_report.sample_outcomes:
    d_durations = [o['duration_seconds'] for o in deepseek_report.sample_outcomes if o['status']=='ok']
    proj = _project(d_durations, total_target, 'deepseek_ocr')
    if proj:
        projections.append(proj)
        print(f'DeepSeek-OCR projection over {total_target} preprocessed pages:')
        print(f'  per-page median {proj["per_page_seconds_median"]}s  →  {proj["projected_hours_median"]}h total')
        print(f'  per-page mean   {proj["per_page_seconds_mean"]}s  →  {proj["projected_hours_mean"]}h total')

if not projections:
    print('No runtime samples in this run. Enable RUN_PADDLE or RUN_DEEPSEEK to project.')

No runtime samples in this run. Enable RUN_PADDLE or RUN_DEEPSEEK to project.


In [22]:
artifact = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'config': {
        'run_full': RUN_FULL,
        'max_pages': MAX_PAGES,
        'run_paddle': RUN_PADDLE,
        'run_deepseek': RUN_DEEPSEEK,
        'recompute': RECOMPUTE,
        'roles': list(ROLES),
        'document_id': DOCUMENT_ID,
    },
    'corpus_totals': totals,
    'extraction_summary': extract_summary,
    'runtime_projections': projections,
    'paddle_report': paddle_report.to_dict() if paddle_report else None,
    'deepseek_report': deepseek_report.to_dict() if deepseek_report else None,
}
out_path = ARTIFACT_DIR / 'extraction.json'
out_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2))
print(f'Wrote {out_path} ({out_path.stat().st_size:,} bytes)')

Wrote /Users/mohasani/Ancient/notebooks/_artifacts/03_dual_extraction/extraction.json (1,437 bytes)


## Next steps

### Local long-running PaddleOCR (flight / offline)

Use the standalone script `scripts/run_paddle_ocr.py` instead of this
notebook for the multi-hour run — it keeps the engine in memory across
the entire corpus and survives SIGTERM gracefully:

```bash
mkdir -p logs
caffeinate -dimsu uv run python scripts/run_paddle_ocr.py \
    --log-file logs/paddle_ocr.log \
    --report-file logs/paddle_ocr_report.json \
    --roles body \
    --verbose
```

Tail progress from another shell with `tail -f logs/paddle_ocr.log`.
Killing the script with `Ctrl-C` finishes the current page then exits;
re-running picks up where it stopped.

### Online DeepSeek-OCR (after landing)

Re-run this notebook with `RUN_PADDLE=0 RUN_DEEPSEEK=1 RUN_FULL=1`, or
use the same orchestrator from a script:

```bash
RUN_PADDLE=0 RUN_DEEPSEEK=1 RUN_FULL=1 \
    uv run jupyter nbconvert --to notebook --execute \
    notebooks/03_dual_extraction.ipynb \
    --output 03_dual_extraction.executed.ipynb \
    --ExecutePreprocessor.timeout=-1
```

### Fusion (after both engines have run)

Run `03b_fusion.ipynb` to merge `paddleOcrText` + `deepseekOcrText`
into the authoritative `textFused`, then re-detect language on the
fused text.